In [2]:
import networkx as nx
from py2neo import Graph, Node, Relationship
import argparse
from urllib.parse import urlparse
import pandas as pd
import os
from tqdm import tqdm

def load_gexf_to_neo4j(
    gexf_file,
    neo4j_uri="bolt://localhost:7687",
    neo4j_user="neo4j",
    neo4j_password="password",
    batch_size=100,
    clear_database=False
):
    """
    Load a GEXF file into Neo4j database.

    Parameters:
    - gexf_file: Path to the GEXF file
    - neo4j_uri: URI for Neo4j connection
    - neo4j_user: Neo4j username
    - neo4j_password: Neo4j password
    - batch_size: Number of operations to batch before committing
    - clear_database: Whether to clear the database before loading
    """
    # Connect to Neo4j
    print(f"Connecting to Neo4j at {neo4j_uri}...")
    graph_db = Graph(neo4j_uri, auth=(neo4j_user, neo4j_password))

    # Clear database if requested
    if clear_database:
        print("Clearing database...")
        graph_db.run("MATCH (n) DETACH DELETE n")

    # Load the NetworkX graph from the GEXF file
    print(f"Loading graph from {gexf_file}...")
    nx_graph = nx.read_gexf(gexf_file)

    # Create constraint for faster lookups (if it doesn't exist)
    try:
        graph_db.run("CREATE CONSTRAINT url_constraint FOR (p:Page) REQUIRE p.url IS UNIQUE")
    except:
        # Constraint may already exist
        pass

    # Create nodes
    print(f"Creating {len(nx_graph.nodes)} nodes in Neo4j...")
    node_batch = []

    for i, (url, attrs) in enumerate(tqdm(nx_graph.nodes(data=True))):
        # Parse URL components
        parsed = urlparse(url)
        domain = attrs.get('domain', parsed.netloc)
        path = attrs.get('path', parsed.path)

        # Create node properties
        properties = {
            'url': url,
            'domain': domain,
            'path': path,
            'scheme': parsed.scheme,
            'query': parsed.query
        }

        # Add any additional attributes from the graph
        for key, value in attrs.items():
            if key not in properties:
                properties[key] = value

        # Create Cypher query
        query = """
        MERGE (p:Page {url: $url})
        SET p.domain = $domain,
            p.path = $path,
            p.scheme = $scheme,
            p.query = $query
        """

        # Add parameters
        node_batch.append(properties)

        # Execute in batches
        if len(node_batch) >= batch_size or i == len(nx_graph.nodes) - 1:
            # Execute batch
            tx = graph_db.begin()
            for params in node_batch:
                tx.run(query, params)
            tx.commit()
            node_batch = []

    # Create relationships
    print(f"Creating {len(nx_graph.edges)} relationships in Neo4j...")
    edge_batch = []

    for i, (source, target, attrs) in enumerate(tqdm(nx_graph.edges(data=True))):
        # Create relationship properties
        properties = {
            'source': source,
            'target': target
        }

        # Add any additional attributes from the edge
        for key, value in attrs.items():
            properties[key] = value

        # Create Cypher query
        query = """
        MATCH (source:Page {url: $source})
        MATCH (target:Page {url: $target})
        MERGE (source)-[r:LINKS_TO]->(target)
        """

        # Add parameters
        edge_batch.append(properties)

        # Execute in batches
        if len(edge_batch) >= batch_size or i == len(nx_graph.edges) - 1:
            # Execute batch
            tx = graph_db.begin()
            for params in edge_batch:
                tx.run(query, params)
            tx.commit()
            edge_batch = []

    # Create domain-level graph
    print("Creating domain-level aggregation...")
    graph_db.run("""
    MATCH (p:Page)
    MERGE (d:Domain {name: p.domain})
    MERGE (p)-[:BELONGS_TO]->(d)
    """)

    # Create domain relationships
    graph_db.run("""
    MATCH (p1:Page)-[:LINKS_TO]->(p2:Page),
          (p1)-[:BELONGS_TO]->(d1:Domain),
          (p2)-[:BELONGS_TO]->(d2:Domain)
    WHERE d1 <> d2
    MERGE (d1)-[r:DOMAIN_LINKS_TO]->(d2)
    ON CREATE SET r.count = 1
    ON MATCH SET r.count = r.count + 1
    """)

    print("Creating indexes for better performance...")
    try:
        graph_db.run("CREATE INDEX domain_index FOR (d:Domain) ON (d.name)")
        graph_db.run("CREATE INDEX page_domain_index FOR (p:Page) ON (p.domain)")
    except:
        # Indexes may already exist
        pass

    print("Done loading data into Neo4j!")

    # Return some stats
    nodes_count = graph_db.run("MATCH (p:Page) RETURN count(p) as count").data()[0]['count']
    rels_count = graph_db.run("MATCH ()-[r:LINKS_TO]->() RETURN count(r) as count").data()[0]['count']
    domains_count = graph_db.run("MATCH (d:Domain) RETURN count(d) as count").data()[0]['count']

    print(f"Statistics:")
    print(f"- Pages: {nodes_count}")
    print(f"- Links between pages: {rels_count}")
    print(f"- Domains: {domains_count}")

    return {
        'pages': nodes_count,
        'links': rels_count,
        'domains': domains_count
    }

def export_sample_cypher_queries(output_file="neo4j_queries.txt"):
    """
    Export sample Cypher queries to analyze the web graph in Neo4j.
    """
    queries = [
        {
            "name": "Top pages by incoming links",
            "query": """
            MATCH (p:Page)<-[r:LINKS_TO]-()
            RETURN p.url as url, count(r) as incoming_links
            ORDER BY incoming_links DESC
            LIMIT 10
            """
        },
        {
            "name": "Top pages by outgoing links",
            "query": """
            MATCH (p:Page)-[r:LINKS_TO]->()
            RETURN p.url as url, count(r) as outgoing_links
            ORDER BY outgoing_links DESC
            LIMIT 10
            """
        },
        {
            "name": "Pages without outgoing links",
            "query": """
            MATCH (p:Page)
            WHERE NOT (p)-[:LINKS_TO]->()
            RETURN p.url as url, p.domain as domain
            LIMIT 100
            """
        },
        {
            "name": "Pages without incoming links",
            "query": """
            MATCH (p:Page)
            WHERE NOT ()-[:LINKS_TO]->(p)
            RETURN p.url as url, p.domain as domain
            LIMIT 100
            """
        },
        {
            "name": "Domain relationship strength",
            "query": """
            MATCH (d1:Domain)-[r:DOMAIN_LINKS_TO]->(d2:Domain)
            RETURN d1.name as source_domain,
                   d2.name as target_domain,
                   r.count as link_count
            ORDER BY link_count DESC
            LIMIT 20
            """
        },
        {
            "name": "Page relationships by domain",
            "query": """
            MATCH (p1:Page)-[:LINKS_TO]->(p2:Page)
            WHERE p1.domain = 'example.com' AND p2.domain = 'blog.example.com'
            RETURN p1.url as source_url, p2.url as target_url
            LIMIT 50
            """
        },
        {
            "name": "Shortest path between two pages",
            "query": """
            MATCH path = shortestPath(
              (start:Page {url: 'https://example.com/'})-[:LINKS_TO*]->(end:Page {url: 'https://example.com/about'})
            )
            UNWIND nodes(path) as page
            RETURN page.url
            """
        },
        {
            "name": "Pages with highest PageRank",
            "query": """
            CALL gds.pageRank.stream('webgraph')
            YIELD nodeId, score
            MATCH (p:Page) WHERE id(p) = nodeId
            RETURN p.url as url, score
            ORDER BY score DESC
            LIMIT 10
            """
        }
    ]

    with open(output_file, 'w') as f:
        f.write("# Sample Neo4j Cypher Queries for Web Graph Analysis\n\n")

        for i, query_item in enumerate(queries, 1):
            f.write(f"## {i}. {query_item['name']}\n")
            f.write("```cypher\n")
            f.write(query_item['query'].strip())
            f.write("\n```\n\n")

    print(f"Exported sample queries to {output_file}")

    # Adding instructions for the PageRank query
    with open(output_file, 'a') as f:
        f.write("## Note on PageRank Query\n")
        f.write("Before running the PageRank query, you need to create a graph projection:\n\n")
        f.write("```cypher\n")
        f.write("CALL gds.graph.project(\n")
        f.write("  'webgraph',\n")
        f.write("  'Page',\n")
        f.write("  'LINKS_TO'\n")
        f.write(")\n")
        f.write("```\n\n")
        f.write("Note: This requires the Neo4j Graph Data Science plugin to be installed.\n")

def create_neo4j_visualizations(output_dir="neo4j_visualizations"):
    """
    Generate Cypher queries for creating visualizations in Neo4j Browser.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    visualizations = [
        {
            "name": "domain_network",
            "title": "Domain Network",
            "query": """
            MATCH (d1:Domain)-[r:DOMAIN_LINKS_TO]->(d2:Domain)
            RETURN d1, r, d2
            """
        },
        {
            "name": "page_network_sample",
            "title": "Page Network (Sample)",
            "query": """
            MATCH (p:Page)
            WITH p LIMIT 50
            MATCH (p)-[r:LINKS_TO]->(p2:Page)
            RETURN p, r, p2
            """
        },
        {
            "name": "ego_network",
            "title": "Ego Network (Example)",
            "query": """
            // Replace with an actual URL from your dataset
            MATCH (center:Page {url: 'https://example.com/'})
            MATCH path = (center)-[:LINKS_TO*1..2]-(other:Page)
            RETURN path
            """
        }
    ]

    for viz in visualizations:
        filename = os.path.join(output_dir, f"{viz['name']}.cypher")
        with open(filename, 'w') as f:
            f.write(f"// {viz['title']}\n")
            f.write(viz['query'].strip())

    print(f"Visualization queries saved to {output_dir}/")


In [14]:

gexf_file = "/Users/wnowogorski/PycharmProjects/CHAT_AGH/web_scraping/web_graph.gexf"
NEO4J_URI="neo4j+s://7c8290c6.databases.neo4j.io"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="FdP0biTewY9Ir4LihMS5tMZT0uWm-YpLxT9mMIFPgTQ"
AURA_INSTANCEID="7c8290c6"
AURA_INSTANCENAME="Free instance"


stats = load_gexf_to_neo4j(
    gexf_file,
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USERNAME,
    neo4j_password=NEO4J_PASSWORD,
    batch_size=50,
    clear_database=True
)

export_sample_cypher_queries()

create_neo4j_visualizations()

Connecting to Neo4j at neo4j+s://7c8290c6.databases.neo4j.io...
Clearing database...


ServiceUnavailable: Cannot connect to any known routers

In [8]:
graph_db = Graph(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [9]:
graph_db

ServiceUnavailable: Cannot connect to any known routers

In [13]:
from neo4j import GraphDatabase

with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
    driver.verify_connectivity()

Unable to retrieve routing information


ServiceUnavailable: Unable to retrieve routing information